In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import (
    layers,
    models,
    callbacks,
    utils,
    metrics,
    losses,
    optimizers,
)

In [ ]:
#load the dataset
ds = load_dataset("Mozilla/flickr30k-transformed-captions", split="test")

dict_keys(['image', 'alt_text', 'sentids', 'split', 'img_id', 'filename', 'original_alt_text'])
['Two people with shaggy hair look at their hands while hanging out in the yard.']


In [ ]:
#preprocess the dataset

ex = ds[100]
#print(len(ds))
#print(len(ds[0]))
#print(ex.keys())
#ex["image"].show()
#print(ex["alt_text"])
#print(ex["filename"])
#print(ex["alt_text"][0]) #that is only the string

def gen():
    for ex in ds:
        img = np.array(ex["image"].convert("RGB"))   # (H, W, 3) uint8
        cap = ex["alt_text"][0]                      # captions
        yield img, cap                             # make just work this funct for 1 iteration when requested (funct is a generator)

tf_ds = tf.data.Dataset.from_generator(
    gen,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.string),
    ),
)

def preprocess(img, cap):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    s = tf.minimum(h, w)
    img = tf.image.resize_with_crop_or_pad(img, s, s)
    img64  = tf.image.resize(img, [64, 64],  method="area")
    img256 = tf.image.resize(img, [256, 256], method="area")
    return {"img64": img64, "img256": img256, "caption": cap}

BATCH = 32
pipeline = (
    tf_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

for img, cap in tf_ds.take(1):
    print("image :", img.shape, img.dtype) 
    print("caption :", cap.numpy())          # cap puts in intelligible letters instead of bytes



image : (500, 333, 3) <dtype: 'uint8'>
caption : b'Two people with shaggy hair look at their hands while hanging out in the yard.'


In [65]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape = (batch, dim))
        return z_mean + tf.exp(0.5*z_log_var)*epsilon  # basically sample a single point from the distibution 

In [69]:
#define the VAE model: encoder -> decoder , GAN

encoder_input = layers.Input(shape = (64,64,3),name = "encoder_input")
x = layers.Conv2D(64, (9,9), strides = 2, padding = "same")(encoder_input)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(128, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(256, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(512, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Flatten()(x)
z_mean = layers.Dense(512, name = "z_mean")(x)
z_log_var = layers.Dense(512, name = "z_log_var")(x)
encoder_output = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_input,[z_mean, z_log_var, encoder_output], name = "encoder")






In [15]:
#define the train function for VAE -> train step , losses 

In [16]:
#train the model

In [17]:
#define the diffusion model and text encoder

In [18]:
#define the train function for diffusion and embedding model -> train step , losses ...

In [19]:
#train the model

In [ ]:
#define the function for generation